# Guide 4: Attention: letting tokens look at each other

*How LLMs work* · [Read the guide](https://drvgautam.github.io/primer-on-aiengineering/attention/) · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drvgautam/primer-on-aiengineering/blob/main/code/notebooks/04-attention.ipynb)

This notebook holds every code block from Guide 4, in the guide's order: 10 code cells, 9 of which print an output the guide quotes. Each cell is preceded by the guide paragraph that introduces it, so you can keep the guide open beside the notebook. The saved outputs are the guide's; `tools/verify_notebooks.py` checks they still match.

Run the setup cell first, then the cells in order: like the guide's blocks, they share variables.

**Sections of the guide**

- Why context pays — 1 code cell
- The idea: a soft lookup — 1 code cell
- Where queries, keys and values come from — no code
- Keeping the softmax usable: the square root of $d_k$ — 1 code cell
- No peeking: the causal mask — 3 code cells
- Several heads at once — 1 code cell
- Order: giving tokens a position — 2 code cells
- The price of looking at everything — 1 code cell
- What attention is, and isn't — no code

In [1]:
# Setup: run this cell first. It makes the repository's code/ folder the working directory, so
# every block below runs exactly as printed in the guide ("run from the repository root").
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/drvgautam/primer-on-aiengineering"
if "google.colab" in sys.modules and not any((p / "code" / "notebooks").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/primer-on-aiengineering"], check=True)
    os.chdir("/content/primer-on-aiengineering")
    pins = [l.strip() for l in open("code/requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins,
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu"], check=True)

here = Path.cwd().resolve()
code_dir = next(p / "code" for p in [here, *here.parents] if (p / "code" / "notebooks").is_dir())
os.chdir(code_dir)

# The guide's outputs were produced with these versions, on CPU. Others may differ in the last digits.
import numpy, tiktoken, torch
pins = dict(l.strip().split("==") for l in open("requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch"))
for name, mod in (("numpy", numpy), ("tiktoken", tiktoken), ("torch", torch)):
    have = mod.__version__.split("+")[0]
    note = "" if have == pins[name] else f"  <- the guide used {pins[name]}; outputs may differ slightly (on Colab, restart the runtime and rerun this cell)"
    print(f"{name:9} {have}{note}")
print("working directory:", Path(code_dir.parent.name) / "code")

# %%gpt2 marks blocks the guide shows but doesn't run, because they need GPT-2 small's 548 MB weights.
# To run them, download the weights as series/running-example.py explains, then set:
#     os.environ["PRIMER_RUN_GPT2"] = "1"
from IPython.core.magic import register_cell_magic

@register_cell_magic
def gpt2(line, cell):
    if os.environ.get("PRIMER_RUN_GPT2") == "1":
        get_ipython().run_cell(cell)
    else:
        print("Skipped: needs GPT-2 weights (see the setup cell to enable).")


numpy     2.2.6
tiktoken  0.14.0
torch     2.14.1
working directory: primer-on-aiengineering/code


## Why context pays

The same counting works for longer contexts. For each number of characters of context $k$, the code below counts what follows every distinct context in Guide 1's toy text. It prints $k$, the lowest possible loss, and the number of distinct contexts. The code in this guide uses numpy 2.2.6, the version pinned in the series `requirements.txt` as of October 2026. Its blocks run in order and share variables.

In [2]:
import numpy as np
from collections import Counter
np.set_printoptions(precision=3, floatmode="fixed", suppress=True)
text = ("the cat sat on the mat and the dog sat on the log and the "
        "cat saw the dog then the dog ran to the cat and the cat ran on the mat")
for k in range(6):
    start = max(k, 1)                                  # k = 0 is scored on the same 127 characters as k = 1
    seen = Counter((text[i - k:i], text[i]) for i in range(start, len(text)))   # (context, next character)
    context = Counter()
    for (c, ch), count in seen.items():
        context[c] += count
    floor = -sum(count * np.log(count / context[c]) for (c, ch), count in seen.items()) / sum(seen.values())
    print(k, f"{floor:.3f}", len(context))

0 2.318 1
1 0.884 15
2 0.337 32
3 0.296 46
4 0.212 59
5 0.179 69


## The idea: a soft lookup

The code prints the same scores, weights and output.

In [3]:
def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)              # softmax only sees differences; this stops exp overflowing
    return np.exp(z) / np.exp(z).sum(axis=-1, keepdims=True)
K = np.array([[0., 0.], [2., 0.], [1., 1.]])           # (3, 2) keys of the, cat, sat
V = np.array([[1., 0.], [0., 1.], [1., 1.]])           # (3, 2) values of the, cat, sat
q = np.array([1., 0.])                                 # (2,) query of sat
scores = K @ q                                         # (3,) one score per key
weights = softmax(scores)                              # (3,) sums to 1
print(scores, weights, weights @ V)

[0.000 2.000 1.000] [0.090 0.665 0.245] [0.335 0.910]


## Keeping the softmax usable: the square root of $d_k$

Dot products spread out as vectors get wider, because each slot adds its own product to the sum. The code below draws 10,000 pairs of random vectors whose entries have a variance of 1, and prints the standard deviation of their dot products at widths 4 and 64.

In [4]:
rng = np.random.default_rng(0)
for d_k in (4, 64):
    uv = rng.normal(size=(10000, 2, d_k))              # 10,000 pairs of random vectors of width d_k
    print(d_k, f"{(uv[:, 0] * uv[:, 1]).sum(axis=1).std():.2f}")
rng = np.random.default_rng(0)
top, top_scaled = [], []
for trial in range(2000):
    q64 = rng.normal(size=64)                          # (64,) one random query
    K64 = rng.normal(size=(16, 64))                    # (16, 64) sixteen random keys
    top.append(softmax(K64 @ q64).max())               # the largest of the 16 weights
    top_scaled.append(softmax(K64 @ q64 / np.sqrt(64)).max())
print(f"{np.mean(top):.3f} {np.mean(top_scaled):.3f}")
print(softmax(scores / np.sqrt(2)))

4 1.98
64 8.01
0.843 0.247
[0.140 0.576 0.284]


## No peeking: the causal mask

The row for ` cat` says it may draw from `The` and from itself, and from nothing after. With the mask, the attention weights are $A = \operatorname{softmax}(S + M)$, taken row by row, and the output is $AV$.

In [5]:
def attention(X, WQ, WK, WV, causal=True):
    Q, K, V = X @ WQ, X @ WK, X @ WV                   # (n, d_k), (n, d_k), (n, d_v)
    S = Q @ K.T / np.sqrt(K.shape[1])                  # (n, n) scaled scores
    M = np.triu(np.full(S.shape, -np.inf), k=1) if causal else 0   # (n, n) mask: -inf above the diagonal
    A = softmax(S + M)                                 # (n, n) weights, every row sums to 1
    return A @ V, A                                    # (n, d_v) new vectors, and the weights

To try it we need embeddings. The next block retrains Guide 3's toy table, 12 words by 4 numbers, with the same code and seed squeezed into fewer lines. Recall that it learns by predicting each word's neighbors within 2 positions, and that "cat" and "dog" ended up with a cosine similarity of 0.95. Then it draws random $4 \times 4$ matrices for $W^Q$, $W^K$ and $W^V$ and runs attention on "the cat sat".

In [6]:
words = text.split()
vocab = sorted(set(words))
ids = np.array([vocab.index(w) for w in words])        # (33,) the text as word IDs
pairs = [(ids[i], ids[j]) for i in range(33) for j in range(max(0, i - 2), min(33, i + 3)) if i != j]
inputs, targets = np.array(pairs).T                    # (126,) each: a word, and one of its neighbors
rng = np.random.default_rng(0)
E = rng.normal(0, 0.1, (12, 4))                        # (12, 4) embedding table
W = rng.normal(0, 0.1, (4, 12))                        # (4, 12) output matrix
for step in range(3001):
    dz = softmax(E[inputs] @ W)                        # (126, 12) predicted probabilities
    dz[np.arange(126), targets] -= 1                   # predicted minus correct
    dW = E[inputs].T @ dz / 126                        # (4, 12)
    dE = np.zeros_like(E)
    np.add.at(dE, inputs, dz @ W.T / 126)              # each pair's gradient goes to its word's row
    E -= 0.5 * dE
    W -= 0.5 * dW

def embed(sentence):
    return E[[vocab.index(w) for w in sentence.split()]]   # (n, 4) one row per word
cat, dog = embed("cat dog")
print(f"{cat @ dog / (np.linalg.norm(cat) * np.linalg.norm(dog)):.2f}")
rng = np.random.default_rng(1)
WQ, WK, WV = (rng.normal(0, 0.5, (4, 4)) for _ in range(3))    # (4, 4) each: random, untrained
out, A = attention(embed("the cat sat"), WQ, WK, WV)
print(A, A.sum(axis=1))

0.95
[[1.000 0.000 0.000]
 [0.619 0.381 0.000]
 [0.757 0.140 0.103]] [1.000 1.000 1.000]


There are zeros above the diagonal, and every row sums to 1. The top-left 1.000 is forced, since the first token has nothing but itself to draw from. Everything else is noise from random matrices. Next come the weights for all five words of the running example, lowercased to fit the toy vocabulary.

In [7]:
print(attention(embed("the cat sat on the"), WQ, WK, WV)[1])

[[1.000 0.000 0.000 0.000 0.000]
 [0.619 0.381 0.000 0.000 0.000]
 [0.757 0.140 0.103 0.000 0.000]
 [0.042 0.017 0.899 0.042 0.000]
 [0.081 0.215 0.135 0.488 0.081]]


## Several heads at once

GPT-2 small has $h = 12$ heads of width 64, and $12 \times 64 = 768$ (the head count is in the [released model's](https://huggingface.co/openai-community/gpt2) configuration). The block below runs those shapes on random numbers. It stores each kind of matrix for all 12 heads as one $768 \times 768$ array, where columns 0 to 63 belong to head 1, the next 64 to head 2, and so on. `split` cuts a product into its 12 slices.

In [8]:
n, d_model, h = 5, 768, 12
d_k = d_model // h                                     # 64
M = np.triu(np.full((n, n), -np.inf), k=1)             # (5, 5) the causal mask from the table
rng = np.random.default_rng(0)
X = rng.normal(size=(n, d_model))                      # (5, 768) stand-in for the embedded running example
WQ_all, WK_all, WV_all, WO = (rng.normal(0, 0.02, (d_model, d_model)) for _ in range(4))   # (768, 768) each

def split(Z):
    return Z.reshape(n, h, d_k).transpose(1, 0, 2)     # (5, 768) -> (12, 5, 64): one slice per head
Q, K, V = split(X @ WQ_all), split(X @ WK_all), split(X @ WV_all)
S = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)            # (12, 5, 5) one score grid per head
A = softmax(S + M)                                     # (12, 5, 5)
joined = (A @ V).transpose(1, 0, 2).reshape(n, d_model)    # (12, 5, 64) -> (5, 768) heads side by side
print(Q.shape, S.shape, (A @ V).shape, joined.shape, (joined @ WO).shape)

(12, 5, 64) (12, 5, 5) (12, 5, 64) (5, 768) (5, 768)


## Order: giving tokens a position

Attention as built so far has a blind spot. It's total with the mask off, which is how the code below runs. With the mask on a little order leaks through, as the accordion at the end of this section shows. A token's output is a weighted sum over the tokens, and nothing in a sum records which term came first. Reverse the rows of $X$ and you get the same output rows, reversed. The block below checks that on "the cat saw the dog", and then compares the output for "saw" in that sentence and in "the dog saw the cat".

In [9]:
X1, X2 = embed("the cat saw the dog"), embed("the dog saw the cat")   # (5, 4) each
out1, _ = attention(X1, WQ, WK, WV, causal=False)
out2, _ = attention(X2, WQ, WK, WV, causal=False)
flipped, _ = attention(X1[::-1], WQ, WK, WV, causal=False)       # the same rows in reverse order
print(np.allclose(flipped, out1[::-1]), np.allclose(out1[2], out2[2]))
pos = np.random.default_rng(2).normal(0, 1.0, (5, 4))  # (5, 4) one random vector per position
out1, _ = attention(X1 + pos, WQ, WK, WV, causal=False)
flipped, _ = attention(X1[::-1] + pos, WQ, WK, WV, causal=False)
print(np.allclose(flipped, out1[::-1]), f"{np.abs(flipped - out1[::-1]).max():.3f}")

True True
False 0.749


> *Optional depth: from the guide's accordion "Other ways to encode position".*

A causal mask also leaks some order by itself. With the mask on, each position draws from a different number of earlier tokens, and trained models with no position vectors at all still pick up position from that ([Haviv et al., 2022](https://arxiv.org/abs/2203.16634)). The toy shows the leak. With the mask on and no position vectors, "saw" gets different outputs in the two sentences, because the tokens before it differ.

In [10]:
masked1, _ = attention(X1, WQ, WK, WV)
masked2, _ = attention(X2, WQ, WK, WV)
print(np.allclose(masked1[2], masked2[2]), f"{np.abs(masked1[2] - masked2[2]).max():.3f}")

False 0.020


## The price of looking at everything

Every token is scored against every token, so each head in each layer fills an $n \times n$ grid of scores. Guide 2 said a model spends computation on every token it reads, and this grid shows how fast that cost grows.

In [11]:
n_max = 1024
print(4 * d_model**2, 4 * d_model**2 + 4 * d_model, n_max * d_model)
print(n * n, h * n * n, n_max**2, h * n_max**2)

2359296 2362368 786432
25 300 1048576 12582912


---

← [Guide 3: Embeddings](03-embeddings.ipynb) · [Guide 5: The transformer](05-transformer.ipynb) →

Found a cell that doesn't reproduce the guide? [Open an issue](https://github.com/drvgautam/primer-on-aiengineering/issues/new?template=does-not-reproduce.yml).